# 第五章：LLM 知识图谱自动构建

## 学习目标

- 理解知识图谱自动构建的流程（文本 → 实体/关系 → 图）
- 使用 LLMGraphTransformer 从文本提取实体和关系
- 控制提取行为（指定允许的节点和关系类型）
- 将提取结果写入 Neo4j
- 对比手动构建 vs 自动构建的优劣

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

# 切换为 GLM（Anthropic 兼容接口）：
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="glm-4-plus", base_url=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"])

print("✓ LLM 初始化完成")

In [ ]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

# 清空数据库，确保从零开始
graph.query("MATCH (n) DETACH DELETE n")
print("\u2713 Neo4j 连接成功，数据库已清空")

## 1. 知识图谱构建的挑战

In [ ]:
# \u274c 手动构建：阅读文本，手写 Cypher
text = """
马云于1999年在杭州创立了阿里巴巴集团。阿里巴巴旗下有淘宝、支付宝等产品。
2014年阿里巴巴在纽约证券交易所上市，成为当时全球最大的IPO。
张勇于2015年接任阿里巴巴CEO。
"""

# 需要人工识别实体和关系，然后逐条写入...
print("手动方式需要：")
print("1. 阅读文本，识别实体（马云、阿里巴巴、杭州...）")
print("2. 识别关系（创立、旗下、上市...）")
print("3. 编写 CREATE 语句")
print("4. 对每段新文本重复以上步骤")
print("\n这在大规模数据上完全不可行！")

In [ ]:
# \u274c 手动方式的 Cypher 示例——光这一段话就要写这么多
manual_cypher = """
CREATE (mayun:Person {name: '马云'})
CREATE (alibaba:Company {name: '阿里巴巴集团'})
CREATE (hangzhou:City {name: '杭州'})
CREATE (taobao:Product {name: '淘宝'})
CREATE (alipay:Product {name: '支付宝'})
CREATE (nyse:Exchange {name: '纽约证券交易所'})
CREATE (zhangyong:Person {name: '张勇'})

CREATE (mayun)-[:FOUNDED {year: 1999}]->(alibaba)
CREATE (alibaba)-[:LOCATED_IN]->(hangzhou)
CREATE (taobao)-[:PRODUCT_OF]->(alibaba)
CREATE (alipay)-[:PRODUCT_OF]->(alibaba)
CREATE (alibaba)-[:LISTED_ON {year: 2014}]->(nyse)
CREATE (zhangyong)-[:CEO_OF {since: 2015}]->(alibaba)
"""

print("手动编写的 Cypher（仅一段话）：")
print(manual_cypher)
print("想象一下处理 1000 篇文档...")

### 刚才发生了什么？

我们用一段短文本演示了手动构建知识图谱的过程：

1. **人工阅读**文本，识别出 7 个实体（马云、阿里巴巴、杭州...）
2. **人工判断**实体类型（Person、Company、City...）
3. **人工识别**实体之间的 6 个关系（创立、旗下、上市...）
4. **手写 Cypher** 创建每个节点和关系

这段话只有 3 句话，就需要写 13 条 CREATE 语句。如果要处理成百上千篇文档，手动方式完全不可扩展。

**核心问题**：能不能让 LLM 自动完成"阅读 → 识别实体 → 识别关系"这个过程？

## 2. LLMGraphTransformer：用 LLM 自动提取

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

# 初始化 LLM 图转换器
llm_transformer = LLMGraphTransformer(llm=llm)

# 准备文档
documents = [Document(page_content=text)]

# 提取实体和关系
graph_documents = llm_transformer.convert_to_graph_documents(documents)

print(f"从 {len(documents)} 个文档中提取了：")
print(f"  节点: {len(graph_documents[0].nodes)} 个")
print(f"  关系: {len(graph_documents[0].relationships)} 个")

In [ ]:
# 查看提取的节点
print("提取的节点：")
for node in graph_documents[0].nodes:
    print(f"  [{node.type}] {node.id}")

In [ ]:
# 查看提取的关系
print("提取的关系：")
for rel in graph_documents[0].relationships:
    print(f"  ({rel.source.id}) -[{rel.type}]-> ({rel.target.id})")

### 刚才发生了什么？

三行代码完成了手动方式需要十几分钟的工作：

1. **`LLMGraphTransformer(llm=llm)`** 创建图转换器，内部会构造一个专门的 prompt，指导 LLM 从文本中提取实体和关系
2. **`Document(page_content=text)`** 将原始文本包装为 LangChain 文档对象——这和我们在 LangChain RAG 章节中用的是同一个 `Document` 类
3. **`convert_to_graph_documents(documents)`** 将文档发送给 LLM，LLM 返回结构化的实体和关系数据
4. 返回的 `GraphDocument` 对象包含：
   - **`nodes`**：节点列表，每个节点有 `id`（名称）、`type`（标签）和可选的 `properties`
   - **`relationships`**：关系列表，每个关系有 `source`（起点）、`target`（终点）和 `type`（关系类型）
   - **`source`**：原始文档引用

> 注意：LLM 每次提取的结果可能略有不同（实体命名、关系类型），这是大模型的非确定性特征。后面我们会学习如何通过约束来提高一致性。

## 3. 控制提取行为

不加约束时，LLM 可能会发挥创造力，产出意想不到的节点类型和关系类型。在实际项目中，我们通常希望图的 schema 是可控的。

In [ ]:
# \u2705 指定允许的节点和关系类型
llm_transformer_controlled = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Company", "Product", "City"],
    allowed_relationships=["FOUNDED", "CEO_OF", "PRODUCT_OF", "LOCATED_IN"],
)

graph_docs_controlled = llm_transformer_controlled.convert_to_graph_documents(documents)

print("受控提取的节点：")
for node in graph_docs_controlled[0].nodes:
    print(f"  [{node.type}] {node.id}")

print("\n受控提取的关系：")
for rel in graph_docs_controlled[0].relationships:
    print(f"  ({rel.source.id}) -[{rel.type}]-> ({rel.target.id})")

### 刚才发生了什么？

通过 `allowed_nodes` 和 `allowed_relationships` 参数，我们给 LLM 划定了提取边界：

- **无约束提取**：LLM 自由发挥，可能把"纽约证券交易所"标记为 `Exchange`、`StockExchange` 或其他类型
- **受控提取**：LLM 只能使用我们指定的类型，不符合的实体和关系会被忽略

这大大提高了图的一致性，也让后续查询更可预测。

### LLMGraphTransformer 关键参数

| 参数 | 说明 | 效果 |
|------|------|------|
| `allowed_nodes` | 允许的节点类型列表 | 只提取指定类型的实体 |
| `allowed_relationships` | 允许的关系类型列表 | 只提取指定类型的关系 |
| `node_properties` | 要提取的节点属性 | 除了名称还提取其他属性（如年龄、成立时间） |
| `relationship_properties` | 要提取的关系属性 | 为关系附加额外信息（如时间、数量） |

## 4. 写入 Neo4j

In [ ]:
# 确保数据库是空的
graph.query("MATCH (n) DETACH DELETE n")

# 将提取的图数据写入 Neo4j
graph.add_graph_documents(
    graph_docs_controlled,
    baseEntityLabel=True,   # 给所有节点添加 __Entity__ 标签
    include_source=True,    # 保留源文档引用
)

print("\u2713 图数据写入 Neo4j 完成")

### 刚才发生了什么？

`graph.add_graph_documents()` 将 `GraphDocument` 中的节点和关系批量写入 Neo4j：

- **`baseEntityLabel=True`**：给所有节点额外添加一个 `__Entity__` 标签。这样无论节点是 `Person` 还是 `Company`，都可以通过 `MATCH (n:__Entity__)` 统一查询
- **`include_source=True`**：在图中保留原始文档节点（标签为 `Document`），并用 `MENTIONS` 关系连接到提取出的实体。这对溯源非常有用——可以追查"这个实体是从哪段文本中提取的"

In [ ]:
# 验证：查看写入的数据
print("图中的节点：")
result = graph.query("MATCH (n) RETURN labels(n) AS labels, n.id AS name LIMIT 10")
for r in result:
    print(f"  {r['labels']}: {r['name']}")

print("\n图中的关系：")
result = graph.query("""
    MATCH (a)-[r]->(b) 
    WHERE type(r) <> 'MENTIONS'
    RETURN a.id AS from, type(r) AS rel, b.id AS to 
    LIMIT 10
""")
for r in result:
    print(f"  {r['from']} -[{r['rel']}]-> {r['to']}")

## 5. 从多个文档批量构建

In [ ]:
# 更多文本
texts = [
    "腾讯由马化腾于1998年在深圳创立。腾讯旗下的微信是中国最大的社交平台，月活用户超过13亿。",
    "字节跳动由张一鸣于2012年在北京创立。旗下产品包括抖音和TikTok，是全球最大的短视频平台。",
    "华为由任正非于1987年在深圳创立。华为是全球领先的通信设备制造商，也在智能手机和云计算领域有重要布局。",
]

docs = [Document(page_content=t) for t in texts]

# 批量提取
all_graph_docs = llm_transformer_controlled.convert_to_graph_documents(docs)

total_nodes = sum(len(gd.nodes) for gd in all_graph_docs)
total_rels = sum(len(gd.relationships) for gd in all_graph_docs)
print(f"从 {len(docs)} 个文档中提取了 {total_nodes} 个节点，{total_rels} 个关系")

In [ ]:
# 查看每个文档的提取结果
for i, gd in enumerate(all_graph_docs):
    print(f"\n--- 文档 {i+1} ---")
    print(f"原文: {gd.source.page_content[:40]}...")
    for rel in gd.relationships:
        print(f"  ({rel.source.id}) -[{rel.type}]-> ({rel.target.id})")

In [ ]:
# 写入 Neo4j
graph.add_graph_documents(all_graph_docs, baseEntityLabel=True, include_source=True)
print("\u2713 批量写入完成")

# 验证
result = graph.query("MATCH (n) RETURN count(n) AS 节点数")
print(f"图中共有 {result[0]['节点数']} 个节点")
result = graph.query("MATCH ()-[r]->() RETURN count(r) AS 关系数")
print(f"图中共有 {result[0]['关系数']} 个关系")

### 刚才发生了什么？

我们用 3 段文本一次性构建了一个包含多家公司、创始人、产品和城市的知识图谱：

1. 将多段文本包装为 `Document` 列表
2. `convert_to_graph_documents()` 对每个文档分别调用 LLM 提取
3. `add_graph_documents()` 将所有提取结果写入同一个 Neo4j 数据库

注意：如果不同文档提到了同一个实体（如"深圳"在腾讯和华为的文本中都出现），`add_graph_documents` 会使用 `MERGE` 操作，避免创建重复节点。这是自动构建的一大优势——**跨文档的实体自动关联**。

## 6. 查询自动构建的知识图谱

In [ ]:
# 查询所有公司和创始人
print("所有公司和创始人：")
result = graph.query("""
    MATCH (p:Person)-[:FOUNDED]->(c:Company)
    RETURN p.id AS 创始人, c.id AS 公司
""")
for r in result:
    print(f"  {r['创始人']} \u2192 {r['公司']}")

In [ ]:
# 查询同城公司
print("同城公司：")
result = graph.query("""
    MATCH (c1:Company)-[:LOCATED_IN]->(city:City)<-[:LOCATED_IN]-(c2:Company)
    WHERE c1 <> c2
    RETURN c1.id AS 公司1, c2.id AS 公司2, city.id AS 城市
""")
for r in result:
    print(f"  {r['公司1']} 和 {r['公司2']} 都在 {r['城市']}")

In [ ]:
# 查询所有产品及其所属公司
print("产品归属：")
result = graph.query("""
    MATCH (prod:Product)-[:PRODUCT_OF]->(c:Company)
    RETURN prod.id AS 产品, c.id AS 公司
""")
for r in result:
    print(f"  {r['产品']} \u2192 {r['公司']}")

### 刚才发生了什么？

我们用标准 Cypher 查询了自动构建的知识图谱：

- **公司和创始人**：`(p:Person)-[:FOUNDED]->(c:Company)` 精确匹配创立关系
- **同城公司**：通过共享的 `City` 节点，发现腾讯和华为都在深圳——这个关联在原始文本中并未直接说明，是通过图结构推理出来的
- **产品归属**：查询产品和公司的从属关系

这正是知识图谱的核心价值：**将分散在多个文档中的信息结构化，并支持关系推理**。

## 7. 提取质量优化

LLM 自动提取并不完美，以下是提升质量的实用技巧：

### 技巧 1：使用更强的模型

模型能力直接影响提取质量。如果预算允许，使用更强的模型（如 `qwen-max`）可以显著提升准确率。

In [ ]:
# 使用更强的模型（示例，根据实际情况调整）
llm_strong = ChatOpenAI(
    model="qwen-max",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

transformer_strong = LLMGraphTransformer(
    llm=llm_strong,
    allowed_nodes=["Person", "Company", "Product", "City"],
    allowed_relationships=["FOUNDED", "CEO_OF", "PRODUCT_OF", "LOCATED_IN"],
)

# 用更强模型提取同一段文本
strong_result = transformer_strong.convert_to_graph_documents(documents)

print("更强模型的提取结果：")
for rel in strong_result[0].relationships:
    print(f"  ({rel.source.id}) -[{rel.type}]-> ({rel.target.id})")

### 技巧 2：约束节点和关系类型

前面已经演示过。这是最重要的质量控制手段——提前定义好图的 schema。

### 技巧 3：文本预处理

- 去除无关信息（广告、导航文本、页眉页脚等）
- 将长文档分段，每段控制在合理长度（建议 500-1000 字）
- 过长的文本可能导致 LLM 遗漏后半部分的实体

### 技巧 4：后处理——合并重复实体

LLM 可能将同一个实体提取为不同名称（如"阿里巴巴" vs "阿里巴巴集团" vs "阿里"）。可以用 Cypher 手动合并：

In [ ]:
# 实体合并示例（假设存在重复实体）
merge_cypher = """
// 将 "阿里" 合并到 "阿里巴巴集团"
MATCH (dup {id: '阿里'}), (main {id: '阿里巴巴集团'})
// 将 dup 的所有关系转移到 main
CALL {
    WITH dup, main
    MATCH (dup)-[r]->(target)
    MERGE (main)-[r2:SAME_TYPE]->(target)
    DELETE r
}
DELETE dup
"""

print("实体合并是提取后处理的常见操作")
print("当图规模较大时，可以考虑用 LLM 辅助识别重复实体")

### 常见问题

- **提取结果为空**：检查 LLM 是否正确连接，可以先用 `llm.invoke("hello")` 验证。部分模型对 function calling / tool use 的支持程度不同，如果 `LLMGraphTransformer` 报错，尝试换一个模型。
- **每次提取结果不一致**：这是 LLM 的正常行为。可以通过设置 `temperature=0` 减少随机性，或用 `allowed_nodes` / `allowed_relationships` 约束输出。
- **实体命名不一致**（"马云" vs "Jack Ma"）：在提取前统一文本语言，或在后处理阶段用 Cypher 合并。
- **关系方向反了**（"阿里巴巴 -[FOUNDED]-> 马云"）：这取决于 LLM 的理解。可以在后处理中通过 Cypher 修正，或在 prompt 中更明确地说明期望的方向。

## 8. 完整流程：文本 → 知识图谱 → 问答

将前面学习的所有步骤串联起来：从原始文本出发，自动构建知识图谱，最后用自然语言提问。

In [ ]:
# 刷新 schema（因为新增了节点和关系类型）
graph.refresh_schema()
print("当前图的 Schema：")
print(graph.schema)

In [ ]:
from langchain_neo4j import GraphCypherQAChain

# 创建问答链
qa_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
)

# 问答
questions = [
    "腾讯是谁创立的？",
    "深圳有哪些科技公司？",
    "抖音是哪家公司的产品？",
]

for q in questions:
    response = qa_chain.invoke({"query": q})
    print(f"Q: {q}")
    print(f"A: {response['result']}\n")

### 刚才发生了什么？

我们完成了从**非结构化文本**到**智能问答**的完整流程：

```
原始文本 → LLMGraphTransformer → GraphDocument → Neo4j → GraphCypherQAChain → 自然语言回答
```

1. **文本 → 图数据**：`LLMGraphTransformer` 自动从文本中提取实体和关系
2. **图数据 → Neo4j**：`add_graph_documents()` 写入图数据库
3. **问题 → Cypher → 回答**：`GraphCypherQAChain` 将自然语言问题转换为 Cypher 查询，在图上执行，再将结果转换为自然语言回答

这就是知识图谱的端到端应用：**让非结构化数据变得可查询、可推理**。

## 9. 手动构建 vs 自动构建

| | 手动构建 | LLM 自动构建 |
|--|--|--|
| **速度** | 慢（人工阅读+编码） | 快（批量自动处理） |
| **准确性** | 高（人工审核） | 中等（依赖 LLM 能力） |
| **可扩展性** | 差（线性人力成本） | 好（可并行处理大量文档） |
| **一致性** | 依赖人员 | 可通过 allowed_nodes 约束 |
| **适合场景** | 小规模、高精度需求 | 大规模、快速原型 |
| **成本** | 人力成本高 | API 调用成本 |

**实际项目建议**：先用 LLM 自动构建快速原型，人工审核后修正错误，再迭代优化提取参数。两种方式结合使用效果最好。

## 总结

本章学习了：

- \u2705 `LLMGraphTransformer` 从文本自动提取实体和关系
- \u2705 `GraphDocument` 数据结构（nodes + relationships + source）
- \u2705 `allowed_nodes` / `allowed_relationships` 控制提取行为
- \u2705 `graph.add_graph_documents()` 写入 Neo4j
- \u2705 从多个文档批量构建知识图谱
- \u2705 完整流程：文本 → 提取 → 存储 → 问答

## 下一章预告

**第六章：GraphRAG 图增强检索** —— 纯向量 RAG 遗漏结构化关系信息，纯图查询需要精确的 Cypher。下一章学习 GraphRAG：将图检索和向量检索结合，构建更强大的问答系统。